# Fabric Benchmarking

## Loading Raw Data

In [1]:
import polars as pl
from azure.identity import InteractiveBrowserCredential
import plotly.express as px

In [2]:
pl.Config.set_tbl_rows(20)

polars.config.Config

In [3]:
# Set up a colour pallete for consistency across charts using endjin colour way
my_palette = ["#84BD00", "#E87722", "#41B6E6", "#31114A", "#A72B2A", "#000000"]
px.defaults.color_discrete_sequence = my_palette

# Default template (controls overall styling)
px.defaults.template = "plotly"

# Default dimensions
px.defaults.height = 800

In [4]:
credential = InteractiveBrowserCredential()

In [5]:
token = credential.get_token("https://storage.azure.com/.default")

In [6]:
# Pre-requisities are to create a Fabric Workspace with a lakehouse, putting names here:
WORKSPACE_NAME = "fabric_performance_benchmark_workspace"
LAKEHOUSE_NAME = "fabric_performance_benchmark_lakehouse"

In [7]:
# Helper function to create base ABFSS path based on workspace and lakehouse name
def construct_base_abfss_path(workspace_name: str, lakehouse_name: str) -> str:
    """Construct the base ABFSS path for a given workspace and lakehouse."""
    # Because it is a URL, replace spaces with %20
    workspace_name = workspace_name.replace(" ", "%20")
    lakehouse_name = lakehouse_name.replace(" ", "%20")
    return f"abfss://{workspace_name}@onelake.dfs.fabric.microsoft.com/{lakehouse_name}.Lakehouse"

# Helper function to create storage options that enable data tools to authenticate and interact with onelake storage
def create_storage_options() -> dict:
    return {
        "bearer_token": token.token,
        "use_fabric_endpoint": "true"
    }

In [8]:
benchmarks_path = f"{construct_base_abfss_path(WORKSPACE_NAME, LAKEHOUSE_NAME)}/Tables/benchmark_repository/benchmarks"

stages_path = f"{construct_base_abfss_path(WORKSPACE_NAME, LAKEHOUSE_NAME)}/Tables/benchmark_repository/stages"

configurations_path = f"{construct_base_abfss_path(WORKSPACE_NAME, LAKEHOUSE_NAME)}/Tables/benchmark_repository/configurations"

benchmark_analytics_path = f"{construct_base_abfss_path(WORKSPACE_NAME, LAKEHOUSE_NAME)}/Tables/benchmark_repository/benchmark_analytics"

In [9]:
storage_options = create_storage_options()

In [11]:
benchmarks = pl.read_delta(benchmark_analytics_path, storage_options=storage_options)

In [12]:
benchmarks["configuration"].unique().to_list()

['02 executors 08/08 cores 56g/56g memory',
 '16 vCores',
 '04 vCores',
 '01 executors 04/04 cores 28g/28g memory',
 '32 vCores',
 '04 executors 04/04 cores 28g/28g memory',
 '02 vCores',
 '02 executors 04/04 cores 28g/28g memory',
 '04 executors 08/08 cores 56g/56g memory',
 '08 vCores',
 '01 executors 08/08 cores 56g/56g memory']

In [13]:
# Filter to only the configuration we care about for this analysis, as there are some other runs in the data that are not relevant
benchmarks = (
    benchmarks
    .filter(pl.col("configuration").is_in(
            [
                # 1 CU
                '02 vCores',
                # 2 CU
                '04 vCores',
                # 4 CU
                '08 vCores',
                '01 executors 04/04 cores 28g/28g memory',
                # 6 CU
                # '02 executors 04/04 cores 28g/28g memory',
                # 8 CU
                '16 vCores',
                '01 executors 08/08 cores 56g/56g memory',
                # 10 CU
                # '04 executors 04/04 cores 28g/28g memory',
                # 12 CU
                '02 executors 08/08 cores 56g/56g memory',
                # 16 CU
                '32 vCores',
                # 20 CU
                '04 executors 08/08 cores 56g/56g memory',
            ]
        )
    )
)

In [14]:
benchmarks.sample(5)

order,platform,configuration,workload_name,run_timestamp,stage_name,stage_time,cpu_count,cpu_usage,memory,memory_usage,stage_time_delta,stage_order,phase,phase_order,configuration_scale,total_v_cores,cu_per_second,cumulative_time
i32,str,str,str,str,str,datetime[μs],i64,f64,f64,f64,f64,i64,str,i64,str,i64,i64,f64
6,"""Fabric Python Notebook""","""16 vCores""","""polars_benchmark""","""20260204_140957""","""write_dates""",2026-02-04 14:13:11.462610,16,2.9,125.54343,8.9,2.174,7,"""create_and_write""",3,"""40""",16,8,194.46
9,"""Fabric Python Notebook""","""02 vCores""","""polars_benchmark""","""20260204_134347""","""read_dates""",2026-02-04 13:46:23.033581,2,15.7,15.36681,51.9,0.582,9,"""read_and_summarise""",4,"""10""",2,1,156.03
7,"""Fabric Python Notebook""","""32 vCores""","""polars_benchmark""","""20260204_142620""","""write_locations""",2026-02-04 14:29:41.683188,32,16.6,251.446178,5.1,2.571,6,"""create_and_write""",3,"""50""",32,16,201.681
1,"""Fabric Python Notebook""","""04 vCores""","""polars_benchmark""","""20260204_134812""","""start""",2026-02-04 13:48:12,4,12.6,31.092934,6.8,null,1,"""start_and_setup""",1,"""20""",4,2,null
9,"""Fabric PySpark Notebook""","""04 executors 08/08 cores 56g/5…","""pyspark_benchmark""","""20260205_180530""","""read_dates""",2026-02-05 18:10:23.329125,8,29.6,62.545181,30.3,0.88,9,"""read_and_summarise""",4,"""55""",40,20,293.33


## Data Source

The use case is implmented using open data provided by the [UK Land Registry House Price Data open data repository](https://www.gov.uk/government/statistical-data-sets/price-paid-data-downloads).

This data is made available for us under an [Open Government Licence](https://www.nationalarchives.gov.uk/doc/open-government-licence/version/3/).

The data is provided as a set of CSV files, one for year calendar year, which have been downloaded ont a Farbric lakehouse.

The data has been collected since 1995, with circa 1 million property sales per year on average, all 30 years of historic data is ~30 million rows for data and ~5GB of raw CSV data.

This is a typical dataset that we encounter for common enterprise client use cases.  The data we are working with will fit in memory for all of the platform configurations we will be testing.  We find that many benchmarks focus on processing 

The objective is focus on the constraints we more often find we are working with:

- Developer exerperience - having processes that run rapidly unlocks significant benefits during the development phase: test suites run quicker, the inner development loop is optimised, time to value is accelerated.  In today's rapidly evolving environment this can yield significant advantages.

- TCO - the cost of running data pipelines both in terms of financial and environmental impact is becoming a significant factor for many organisations.

## Use Case

The use case mimics a common set of data transformations that you would see on data of this nature.  It includes:

0. Start Up & Set Up - the overhead of provisioning the platform (Spark or Python) on whcih the code is going to run, then completing various tasks such as importing Python packages.
1. Ingestion & Transformation - reading raw data from a set of CSV files, standardising, cleaning and adding new features.
2. Create Dimensional Model - taking different slices of the transformed data and writing that out to the lakehouse in Delta format  data for downstream consumption, in this case as a dimensional model for Power BI.
3. Reading and Summarise - reading the tables back in running analysis based on filtering, joining and summarising the data across different categories.
4. Capture & Clean Up - at various points in the process above a set of benchmark timestamps are captured along with other metadata such as memory consumption.  These are written to permanent storage in the lakehouse for analysis.

A more detailed overview of this process is captured in the mermaid diagram below, with the following symbology:
- ⬆️ - reading from lakehouse.
- 🔧 - data wrangling.
- ⬇️ - write to lakehouse.
- 📊 - points in the process where a benchmark timestamp is captured.


```mermaid
flowchart LR

subgraph Phase0["Phase 0 - Start Up & Set Up"]
    direction TB
    A1[Start Up Platform] --> A2[Import Packages]
    A2 --> A3[Set Up Logging]
    A3 --> A4[Define Constants]
    A4 --> A5[Set Up Helper Functions]
    A5 --> A6[Configure Paths]
    A6 --> A7[Initialise BenchmarkManager]
    A7 --> A8["📊 capture: setup"]
end

subgraph Phase1["Phase 1 - Ingest & Transform"]
    direction TB
    B1["⬆️Scan CSV Files"] --> B2["📊 capture: ingest"]
    B2 --> B3["🔧Transform Data"]
    B3 --> B4["Cache Transformed Data"]
    B4 --> B5["📊 capture: transform"]
end

subgraph Phase2["Phase 2 - Create Dimensional Model"]
    direction TB
    C1[🔧Create Prices Table] --> C2[⬇️Write Prices to Delta]
    C2 --> C3["📊 capture: write_prices"]
    
    C3 --> D1["🔧Create Dates Dimension"]
    D1 --> D2["⬇️Write Dates to Delta"]
    D2 --> D3["📊 capture: write_dates"]
    
    D3 --> E1["🔧Create Locations Dimension"]
    E1 --> E2["⬇️Write Locations to Delta"]
    E2 --> E3["📊 capture: write_locations"]
end

subgraph Phase3["Phase 3 - Read & Summarise"]
    direction TB
    F1[⬆️Read Prices from Delta] --> F2["📊 capture: read_prices"]
    F2 --> F3["⬆️Read Dates from Delta"]
    F3 --> F4["📊 capture: read_dates"]
    F4 --> F5["🔧Join Prices ⟕ Dates"]
    F5 --> F6["🔧Aggregate by Month & Property Type"]
    F6 --> F7["🔧Collect & Display Results"]
    F7 --> F8["📊 capture: join_and_summarise"]
end

subgraph Phase4["Phase 4 - Capture & Clean Up"]
    direction TB
    G1[⬇️Export Benchmarks] --> G2[Calculate Elapsed Time]
    G2 --> G3[Remove Working Data]
end

Phase0 --> Phase1
Phase1 --> Phase2
Phase2 --> Phase3
Phase3 --> Phase4
```

## Workloads

This study was carried out to compare running the common use case implemented across 4 different workloads running on the Fabric Platform:

1. Pandas - the default package for those who first introduced data engineering using Python.  The package has a huge following.  Only suitable for data volumes which can fit into memory.

2. PySpark - the Python API for Apache Spark, a common choice for enterprise platform for data engineering.  Made popular by Databricks and Azure Synapse which provide Spark as cloud PaaS.  Spark is a distributed compute platform which can scale up to handle true "big data" workloads.

3. Polars - a Rust engine with a Python API which provides a powerful query engine with a dataframe based API.  See blog post X for more details.

4. DuckDB - a C++ engine with a Python API which provides an analytical database engine.  See blog post Y for more details.

## Platforms

Fabric offers mutliple compute platforms.  For this study we leveraged two:

- Spark notebooks - a notebook experience over a Spark cluster hosted on Fabric.  Enables polyglot development (Python, R, SQL) over a Spark cluster which is spun up according to your chosen configuration (vCores, memory, number of executor nodes) on demand.

- Python notebooks - a relatively new addition to Fabric.  Python notebooks provide a single node for execution which can be sized according to range of pre-defined configurations (vCores and memory).  Whilst they are designed for "smaller" workloads, we find that the majority of enterprise use cases can be accomodated on this platform through intelligent choice of tooling and design.


## Configurations

It is difficult to achieve parity across the Spark and Python notebook platforms.  We opted for the following configurations which we labelled using "T Shirt Sizes" for ease of cross comparison:

| T-Shirt Size | Python Notebook Configuration | Spark Pool Configuration |
| --- | ---                 | --- |
| XS  | 2 vCores, 16G RAM   |  |
| S   | 4 vCores, 32G RAM   | 1 Executor 4/4 vCores 28G/28G RAM |
| M   | 8 vCores, 64G RAM  | 1 Executor 8/8 vCores 56G/56G RAM |
| L   | 16 vCores, 128G RAM | 2 Executors 8/8 vCores 56G/56G RAM |
| XL  | 32 vCores, 256G RAM | 4 executors 8/8 vCores 56G/56G RAM |

The number of runs per configuration of environment is provided below:

In [15]:
(
    benchmarks
    .group_by(["platform", "configuration"])
    .agg(
        pl.col("cpu_count").max().alias("cpu_count"),
        pl.col("memory").max().alias("memory")
    )
    .sort(["platform", "configuration"])
)

platform,configuration,cpu_count,memory
str,str,i64,f64
"""Fabric PySpark Notebook""","""01 executors 04/04 cores 28g/2…",8,62.545181
"""Fabric PySpark Notebook""","""01 executors 08/08 cores 56g/5…",8,62.545181
"""Fabric PySpark Notebook""","""02 executors 08/08 cores 56g/5…",8,62.545181
"""Fabric PySpark Notebook""","""04 executors 08/08 cores 56g/5…",8,62.545181
"""Fabric Python Notebook""","""02 vCores""",2,15.36681
"""Fabric Python Notebook""","""04 vCores""",4,31.092934
"""Fabric Python Notebook""","""08 vCores""",8,62.545181
"""Fabric Python Notebook""","""16 vCores""",16,125.54343
"""Fabric Python Notebook""","""32 vCores""",32,251.446178


## Methodology

Multiple runs were completed for each combination of: Platform, Workload and Configuration to enable median times to be calculated.

## Fabric Default Environments

The default configuration for a Pyspark notebook is `1 Executor 8/8 vCores 56G/56G RAM`.

The default configuration for a Python notebook is `2 vCores, 16G RAM`.

This means that the spin up time for these configurations of platform were significantly faster as we can see in the analysis below:

## Analysis

## Overall Results

In [16]:
overall_benchmarks = (
    benchmarks
    .sort(["run_timestamp", "order"])
    .group_by(["platform", "configuration", "workload_name", "run_timestamp", "cu_per_second"])
    .agg(pl.col("stage_time_delta").sum().alias("total_elapsed_time"))
    .sort(["platform", "configuration", "workload_name", "run_timestamp"])
)

In [17]:
fig = px.box(overall_benchmarks, 
             x="workload_name",
             y="total_elapsed_time",
             color="workload_name",
             title="Total Elapsed Time (including Setup)",
             facet_col="cu_per_second",
             category_orders={
                 "cu_per_second": [1, 2, 4, 8, 12, 16, 20],
                 "workload_name": ["pandas_benchmark", "pyspark_benchmark", "polars_benchmark", "duckdb_benchmark"]
                 }
             )

fig.update_layout(
    yaxis_title="Total Elapsed Time (seconds)"
)

fig.update_xaxes(visible=False)

fig.show()

In [18]:
overall_benchmarks_excluding_start_and_setup = (
    benchmarks
    .filter(pl.col("phase") != "start_and_setup")
    .sort(["run_timestamp", "order"])
    .group_by(["platform", "configuration", "workload_name", "run_timestamp", "cu_per_second"])
    .agg(pl.col("stage_time_delta").sum().alias("total_execution_time"))
    .sort(["platform", "configuration", "workload_name", "run_timestamp"])
)

In [19]:
fig = px.box(overall_benchmarks_excluding_start_and_setup, 
             x="workload_name",
             y="total_execution_time",
             color="workload_name",
             title="Total Execution Time (excluding Setup)",
             facet_col="cu_per_second",
             category_orders={
                 "cu_per_second": [1, 2, 4, 8, 12, 16, 20],
                 "workload_name": ["pandas_benchmark", "pyspark_benchmark", "polars_benchmark", "duckdb_benchmark"]
                 }
             )

fig.update_layout(
    yaxis_title="Total Execution Time (seconds)"
)

fig.update_xaxes(visible=False)

fig.show()

In [20]:
fig = px.box(overall_benchmarks_excluding_start_and_setup, 
             x="cu_per_second",
             y="total_execution_time",
             color="workload_name",
             title="Total Execution Time (excluding Setup)",
             facet_col="workload_name",
             category_orders={
                 "cu_per_second": [1, 2, 4, 8, 12, 16, 20],
                 "workload_name": ["pandas_benchmark", "pyspark_benchmark", "polars_benchmark", "duckdb_benchmark"]
                 }
             )

fig.update_layout(
    yaxis_title="Total Execution Time (seconds)"
)


fig.show()

In [21]:
fig = px.box(overall_benchmarks_excluding_start_and_setup, 
             x="cu_per_second",
             y="total_execution_time",
             color="workload_name",
             title="Total Execution Time (excluding Setup)",
             category_orders={
                 "cu_per_second": [1, 2, 4, 8, 12, 16, 20],
                 "workload_name": ["pandas_benchmark", "pyspark_benchmark", "polars_benchmark", "duckdb_benchmark"]
                 }
             )

fig.update_layout(
    yaxis_title="Total Execution Time (seconds)"
)


fig.show()

## CU Cost Analysis


In [22]:
cu_cost = (
    overall_benchmarks_excluding_start_and_setup
    .group_by(["platform", "configuration", "workload_name", "cu_per_second"])
    .agg(pl.col("total_execution_time").median().alias("median_execution_time"))
    .with_columns(
        (pl.col("median_execution_time") * pl.col("cu_per_second")).alias("total_cu_cost") )
    .sort(["platform", "configuration", "workload_name", "cu_per_second"])
)

In [23]:
fig = px.bar(cu_cost, 
             x="workload_name",
             y="total_cu_cost",
             color="workload_name",
             title="Execution Cost",
             facet_col="cu_per_second",
             category_orders={
                 "cu_per_second": [1, 2, 4, 8, 12, 16, 20],
                 "workload_name": ["pandas_benchmark", "pyspark_benchmark", "polars_benchmark", "duckdb_benchmark"]
                 }
             )

fig.update_layout(
    yaxis_title="Total CU Cost"
)

fig.update_xaxes(visible=False)

fig.show()

In [24]:
cu_cost.columns

['platform',
 'configuration',
 'workload_name',
 'cu_per_second',
 'median_execution_time',
 'total_cu_cost']

In [25]:
overall_benchmarks_median = (
    overall_benchmarks
    .group_by(["platform", "configuration", "workload_name", "cu_per_second"])
    .agg(pl.col("total_elapsed_time").median().alias("median_total_elapsed_time"))
    .sort(["platform", "configuration", "workload_name", "cu_per_second"])
)

In [26]:
overall_benchmarks_median.columns

['platform',
 'configuration',
 'workload_name',
 'cu_per_second',
 'median_total_elapsed_time']

In [27]:
elapsed_time_cost = (
    overall_benchmarks_median
    .join(cu_cost, on=["platform", "configuration", "workload_name", "cu_per_second"])
)

In [28]:
fig = px.scatter(
    elapsed_time_cost,
    x="median_total_elapsed_time",
    y="total_cu_cost",
    log_y=True,
    color="workload_name",
    symbol="workload_name",
    title="Execution Cost vs Time",
    labels={
        "median_total_time": "Median Time (s)",
        "total_cu_cost": "Total CU Cost"
    },
    category_orders={
        "workload_name": ["pandas_benchmark", "pyspark_benchmark", "polars_benchmark", "duckdb_benchmark"]
    }
)

fig.update_traces(marker=dict(size=12))  # default is 6

fig.show()

## Stage Analysis

In [29]:
stage_cumulative_time = (
    benchmarks
    .filter(pl.col("phase") != "start_and_setup")
    .sort(["run_timestamp", "order"])
    .group_by(["platform", "configuration", "workload_name", "cu_per_second", "stage_name", "order"])
    .agg(pl.col("cumulative_time").median().alias("median_cumulative_time"))
    .sort(["platform", "configuration", "workload_name", "cu_per_second", "order"])
)  

In [30]:
fig = px.line(
    stage_cumulative_time,
    x="stage_name", 
    y="median_cumulative_time",
    color="workload_name",  # Different line color per workload_name
    facet_col="cu_per_second",  # Different dash pattern per t_shirt_size
    markers=True,
    title="Median Cumulative Time by Stage and Configuration",
    category_orders={
                 "cu_per_second": [1, 2, 4, 8, 12, 16, 20],
                 "workload_name": ["pandas_benchmark", "pyspark_benchmark", "polars_benchmark", "duckdb_benchmark"],
                 "stage_name": ["ingest", "transform", "write_prices", "write_dates", "write_locations", "read_prices", "read_dates", "join_and_summarise"]
                 }
)
fig.show()

In [31]:
stage_max_memory = (
    benchmarks
    .sort(["run_timestamp", "order"])
    .group_by(["platform", "configuration", "workload_name", "cu_per_second", "stage_name", "order"])
    .agg(pl.col("memory_usage").max().alias("max_memory_usage"))
    .sort(["platform", "configuration", "workload_name", "cu_per_second", "order"])
)  

In [32]:
fig = px.line(
    stage_max_memory,
    x="stage_name", 
    y="max_memory_usage",
    range_y=[0, 100],
    color="workload_name",  # Different line color per workload_name
    facet_col="cu_per_second",  # Different dash pattern per t_shirt_size
    markers=True,
    title="Memory Usage by Stage and Configuration",
    category_orders={
                 "cu_per_second": [1, 2, 4, 8, 12, 16, 20],
                 "workload_name": ["pandas_benchmark", "pyspark_benchmark", "polars_benchmark", "duckdb_benchmark"],
                 "stage_name": ["start", "setup", "ingest", "transform", "write_prices", "write_dates", "write_locations", "read_prices", "read_dates", "join_and_summarise"]
                 }
)
fig.show()